# RLAD Noise-Control Experiment (Hendrycks MATH): Track A (Clean) vs Track B (Mixed) — Qwen3-1.7B, 400-Step Analysis

This notebook analyzes the results of the Qwen3-1.7B Hendrycks MATH run of the RLAD noise-control experiment, comparing:
- **Track A (clean):** Trained on standard Hendrycks MATH data (Completed 400 steps)
- **Track B (mixed):** Trained on Hendrycks MATH with prepended trivia facts (Completed 400 steps)

Both tracks use identical hyperparameters and evaluate on the same clean test set. This analysis covers the entire 400 training steps for both models.

All data is fetched directly from WandB and Modal AI artifacts — no external CSV files required.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import seaborn as sns
import wandb
import os

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Fetch histories directly from WandB
PROJECT = "cfw20-stanford-university/rlad-hendrycks"
RUN_ID_CLEAN = "5ot5gmgq"   # qwen3-1p7b-hendrycks-grpo-clean
RUN_ID_MIXED = "00h4z032"  # qwen3-1p7b-hendrycks-grpo-mixed

api = wandb.Api()
run_a = api.run(f"{PROJECT}/{RUN_ID_CLEAN}")
run_b = api.run(f"{PROJECT}/{RUN_ID_MIXED}")

# Pull full histories as DataFrames
df_a = run_a.history(samples=10000, pandas=True)
df_b = run_b.history(samples=10000, pandas=True)

# Pull summary dicts
summary_a = dict(run_a.summary)
summary_b = dict(run_b.summary)

print(f"Track A (clean):  {run_a.name} ({RUN_ID_CLEAN})")
print(f"Track B (mixed):  {run_b.name} ({RUN_ID_MIXED})")
print()
print(f"Track A: {len(df_a)} rows, steps: {df_a['_step'].min():.0f} - {df_a['_step'].max():.0f}")
print(f"Track B: {len(df_b)} rows, steps: {df_b['_step'].min():.0f} - {df_b['_step'].max():.0f}")

## 0. Dataset Characteristics & Prompt Inspection

Before diving into training dynamics, let's ground ourselves in the raw data. This section loads the generated parquet files used by both tracks and shows exactly what the model sees at train time.

In [ ]:
import os
import pandas as pd
import json

DATA_DIR = "data_hendrycks_math"

df_train_clean = pd.read_parquet(os.path.join(DATA_DIR, "train_clean.parquet"))
df_train_mixed = pd.read_parquet(os.path.join(DATA_DIR, "train_mixed.parquet"))
df_test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

print("Dataset shapes:")
print(f"  train_clean : {df_train_clean.shape}")
print(f"  train_mixed : {df_train_mixed.shape}")
print(f"  test        : {df_test.shape}")

### 0.1 Dataset Inventory

We report row counts and question-length statistics for each split. Lengths are computed on `extra_info.question` (for the mixed split this includes the prepended trivia fact).

In [ ]:
def question_stats(df, label):
    questions = df["extra_info"].apply(lambda x: x["question"])
    char_counts = questions.str.len()
    token_counts = questions.apply(lambda q: len(q.split()))
    return {
        "split": label,
        "rows": len(df),
        "char_mean": round(char_counts.mean(), 1),
        "char_median": round(char_counts.median(), 1),
        "char_min": int(char_counts.min()),
        "char_max": int(char_counts.max()),
        "token_mean": round(token_counts.mean(), 1),
        "token_median": round(token_counts.median(), 1),
        "token_min": int(token_counts.min()),
        "token_max": int(token_counts.max()),
    }

stats = [
    question_stats(df_train_clean, "train_clean"),
    question_stats(df_train_mixed, "train_mixed"),
    question_stats(df_test, "test"),
]

stats_df = pd.DataFrame(stats)
display(stats_df)

### 0.2 Prompt Spot-Check

Below we show the exact prompt dict for the **same underlying MATH question** in its clean form (Track A) and its trivia-augmented form (Track B). This mirrors what the model sees during training.

In [ ]:
# In train_mixed, first half are clean originals and second half are augmented clones.
n_clean = len(df_train_clean)
mixed_clean = df_train_mixed.iloc[:n_clean]
mixed_aug = df_train_mixed.iloc[n_clean:]

# Pick a representative example
example_idx = 42
clean_row = mixed_clean.iloc[example_idx]
aug_row = mixed_aug.iloc[example_idx]

print("=" * 70)
print("CLEAN PROMPT (Track A)")
print("=" * 70)
print(json.dumps(clean_row["prompt"].tolist(), indent=2, ensure_ascii=False))
print()
print("=" * 70)
print("MIXED PROMPT (Track B) — Same underlying question")
print("=" * 70)
print(json.dumps(aug_row["prompt"].tolist(), indent=2, ensure_ascii=False))

## 1. Learning Curve Comparison (Validation Accuracy)

We compare validation reward (accuracy) over steps for both tracks, aligned to the full step range (0-400).

In [ ]:
# Filter to common step range
max_step = min(df_a['_step'].max(), df_b['_step'].max())
df_a_aligned = df_a[df_a['_step'] <= max_step].copy()
df_b_aligned = df_b[df_b['_step'] <= max_step].copy()

# Find validation reward columns
val_cols_a = [c for c in df_a.columns if 'val/HuggingFaceH4/MATH-500/reward/mean' in c]
val_cols_b = [c for c in df_b.columns if 'val/HuggingFaceH4/MATH-500/reward/mean' in c]

print("Validation columns found:")
print(f"  Track A: {val_cols_a}")
print(f"  Track B: {val_cols_b}")

In [ ]:
# Plot validation accuracy over steps
fig, ax = plt.subplots(figsize=(12, 6))

# Get validation data (non-null values only)
val_a = df_a_aligned[df_a_aligned['val/HuggingFaceH4/MATH-500/reward/mean'].notna()]
val_b = df_b_aligned[df_b_aligned['val/HuggingFaceH4/MATH-500/reward/mean'].notna()]

ax.plot(val_a['_step'], val_a['val/HuggingFaceH4/MATH-500/reward/mean'], 'o-', label='Track A (Clean)', linewidth=2, markersize=6)
ax.plot(val_b['_step'], val_b['val/HuggingFaceH4/MATH-500/reward/mean'], 's-', label='Track B (Mixed)', linewidth=2, markersize=6)

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Validation Accuracy (Reward Mean)', fontsize=12)
ax.set_title('Validation Accuracy: Track A (Clean) vs Track B (Mixed)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Add annotations for final values
if len(val_a) > 0:
    ax.annotate(f"{val_a['val/HuggingFaceH4/MATH-500/reward/mean'].iloc[-1]:.3f}", 
                xy=(val_a['_step'].iloc[-1], val_a['val/HuggingFaceH4/MATH-500/reward/mean'].iloc[-1]),
                xytext=(10, 10), textcoords='offset points', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

if len(val_b) > 0:
    ax.annotate(f"{val_b['val/HuggingFaceH4/MATH-500/reward/mean'].iloc[-1]:.3f}", 
                xy=(val_b['_step'].iloc[-1], val_b['val/HuggingFaceH4/MATH-500/reward/mean'].iloc[-1]),
                xytext=(10, -20), textcoords='offset points', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.savefig('1p7_hendrycks_validation_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print step-by-step comparison
print("\nValidation Accuracy at Common Steps:")
print(f"{'Step':<8} {'Track A':<12} {'Track B':<12} {'Delta (B-A)':<15}")
print("-" * 50)
for step in sorted(set(val_a['_step']) & set(val_b['_step'])):
    a_val = val_a[val_a['_step'] == step]['val/HuggingFaceH4/MATH-500/reward/mean'].values
    b_val = val_b[val_b['_step'] == step]['val/HuggingFaceH4/MATH-500/reward/mean'].values
    if len(a_val) > 0 and len(b_val) > 0:
        delta = b_val[0] - a_val[0]
        print(f"{step:<8} {a_val[0]:<12.4f} {b_val[0]:<12.4f} {delta:<+15.4f}")

## 2. Timing Efficiency Analysis

We analyze step timing to understand why Track A was slower per step than Track B.

In [ ]:
# Plot timing comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

timing_metrics = [
    ('perf/time_per_step', 'Total Time per Step (s)'),
    ('timing_s/gen', 'Generation Time (s)'),
    ('timing_s/update_actor', 'Actor Update Time (s)'),
    ('timing_s/ref', 'Reference Model Time (s)'),
]

for idx, (metric, title) in enumerate(timing_metrics):
    ax = axes[idx // 2, idx % 2]
    
    if metric in df_a_aligned.columns:
        data_a = df_a_aligned[df_a_aligned[metric].notna()]
        ax.plot(data_a['_step'], data_a[metric], 'o-', label='Track A', alpha=0.7)
    
    if metric in df_b_aligned.columns:
        data_b = df_b_aligned[df_b_aligned[metric].notna()]
        ax.plot(data_b['_step'], data_b[metric], 's-', label='Track B', alpha=0.7)
    
    ax.set_xlabel('Step')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Timing Breakdown: Track A vs Track B', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('1p7_hendrycks_timing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print average timings
print("\nAverage Timing Comparison (seconds):")
print(f"{'Metric':<30} {'Track A':<12} {'Track B':<12} {'Ratio (A/B)':<15}")
print("-" * 70)
for metric, title in timing_metrics:
    a_mean = df_a_aligned[metric].mean() if metric in df_a_aligned.columns else np.nan
    b_mean = df_b_aligned[metric].mean() if metric in df_b_aligned.columns else np.nan
    ratio = a_mean / b_mean if b_mean > 0 else np.nan
    print(f"{title:<30} {a_mean:<12.2f} {b_mean:<12.2f} {ratio:<15.2f}")

## 3. Response Length Dynamics

Response length is a proxy for 'rambling.' We compare how quickly each track learned to write concise answers.

In [ ]:
# Plot response length dynamics
fig, ax = plt.subplots(figsize=(12, 6))

# Response length
rl_a = df_a_aligned[df_a_aligned['response_length/mean'].notna()]
rl_b = df_b_aligned[df_b_aligned['response_length/mean'].notna()]

ax.plot(rl_a['_step'], rl_a['response_length/mean'], 'o-', label='Track A (Clean)', linewidth=2)
ax.plot(rl_b['_step'], rl_b['response_length/mean'], 's-', label='Track B (Mixed)', linewidth=2)

# Add clip ratio on secondary y-axis
ax2 = ax.twinx()
clip_a = df_a_aligned[df_a_aligned['response_length/clip_ratio'].notna()]
clip_b = df_b_aligned[df_b_aligned['response_length/clip_ratio'].notna()]

ax2.plot(clip_a['_step'], clip_a['response_length/clip_ratio'], 'o--', 
         color='C0', alpha=0.5, label='Track A Clip Ratio')
ax2.plot(clip_b['_step'], clip_b['response_length/clip_ratio'], 's--', 
         color='C1', alpha=0.5, label='Track B Clip Ratio')

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Mean Response Length', fontsize=12, color='black')
ax2.set_ylabel('Response Length Clip Ratio', fontsize=12, color='gray')
ax.set_title('Response Length Dynamics: Clean vs Mixed Training', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax2.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('1p7_hendrycks_response_length_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
print("\nResponse Length Summary (all steps):")
print(f"Track A: mean={rl_a['response_length/mean'].mean():.1f}, max={rl_a['response_length/mean'].max():.1f}")
print(f"Track B: mean={rl_b['response_length/mean'].mean():.1f}, max={rl_b['response_length/mean'].max():.1f}")
print(f"\nClip Ratio Summary:")
print(f"Track A: mean={clip_a['response_length/clip_ratio'].mean():.3f}, max={clip_a['response_length/clip_ratio'].max():.3f}")
print(f"Track B: mean={clip_b['response_length/clip_ratio'].mean():.3f}, max={clip_b['response_length/clip_ratio'].max():.3f}")

## 4. Reward and Advantage Distribution

We examine training reward statistics to understand exploration vs. exploitation dynamics.

In [ ]:
# Plot additional training dynamics
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

dynamics_metrics = [
    ('critic/rewards/mean', 'Mean Reward'),
    ('actor/entropy', 'Policy Entropy'),
    ('actor/kl_div', 'KL Divergence'),
    ('actor/clip_ratio', 'Policy Clip Ratio'),
    ('global_seqlen/mean', 'Global Sequence Length'),
    ('val/HuggingFaceH4/MATH-500/reward/std', 'Reward Std Dev (Validation)'),
]

for idx, (metric, title) in enumerate(dynamics_metrics):
    ax = axes[idx // 3, idx % 3]
    
    if metric in df_a_aligned.columns:
        data_a = df_a_aligned[df_a_aligned[metric].notna()]
        ax.plot(data_a['_step'], data_a[metric], 'o-', label='Track A', alpha=0.7, markersize=4)
    
    if metric in df_b_aligned.columns:
        data_b = df_b_aligned[df_b_aligned[metric].notna()]
        ax.plot(data_b['_step'], data_b[metric], 's-', label='Track B', alpha=0.7, markersize=4)
    
    ax.set_xlabel('Step')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Dynamics: Track A vs Track B', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('1p7_hendrycks_training_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Memory Utilization

We check GPU memory pressure for both tracks.

In [ ]:
# Memory usage comparison (if available)
memory_metrics = ['perf/max_allocated_memory_gb', 'max_memory_mb']
available_memory = [m for m in memory_metrics if m in df_a_aligned.columns or m in df_b_aligned.columns]

if available_memory:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for metric in available_memory:
        if metric in df_a_aligned.columns:
            data_a = df_a_aligned[df_a_aligned[metric].notna()]
            ax.plot(data_a['_step'], data_a[metric], 'o-', label=f'Track A: {metric}', alpha=0.7)
        
        if metric in df_b_aligned.columns:
            data_b = df_b_aligned[df_b_aligned[metric].notna()]
            ax.plot(data_b['_step'], data_b[metric], 's-', label=f'Track B: {metric}', alpha=0.7)
    
    ax.set_xlabel('Step')
    ax.set_ylabel('Memory (GB or MB)')
    ax.set_title('Peak Memory Usage Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('1p7_hendrycks_memory_usage.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Memory metrics not available in WandB logs.")

## 6. Step-wise Accuracy Gap

We compute the accuracy gap between Track B and Track A at each common validation step.

In [ ]:
# Plot accuracy gap over time
fig, ax = plt.subplots(figsize=(12, 6))

# Filter to common validation steps
common_steps = sorted(set(val_a['_step']) & set(val_b['_step']))
if common_steps:
    gaps = []
    for step in common_steps:
        a_val = val_a[val_a['_step'] == step]['val/HuggingFaceH4/MATH-500/reward/mean'].values
        b_val = val_b[val_b['_step'] == step]['val/HuggingFaceH4/MATH-500/reward/mean'].values
        if len(a_val) > 0 and len(b_val) > 0:
            gaps.append(b_val[0] - a_val[0])
    
    ax.plot(common_steps, gaps, 'o-', linewidth=2, markersize=8, color='purple')
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Accuracy Gap (Mixed - Clean)', fontsize=12)
    ax.set_title('Generalization Gap: Mixed Training vs Clean Training', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Color positive vs negative
    for i, (step, gap) in enumerate(zip(common_steps, gaps)):
        color = 'green' if gap > 0 else 'red'
        ax.plot(step, gap, 'o', color=color, markersize=10, alpha=0.6)
    
    plt.tight_layout()
    plt.savefig('1p7_hendrycks_accuracy_gap.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nAccuracy Gap Summary:")
    print(f"Mean gap: {np.mean(gaps):+.4f}")
    print(f"Max gap (mixed better): {np.max(gaps):+.4f} at step {common_steps[np.argmax(gaps)]}")
    print(f"Min gap (mixed worse):  {np.min(gaps):+.4f} at step {common_steps[np.argmin(gaps)]}")
else:
    print("No common validation steps found.")

## 7. Summary and Conclusions

### Key Findings

1. **Validation Accuracy:** Track B (mixed) achieved higher validation accuracy than Track A (clean) across the entire training run up to step 400.
2. **Training Efficiency:** Track B was faster per step, likely due to learning concise response formatting earlier.
3. **Response Length:** Track B maintained shorter mean response lengths and lower clip ratios, indicating faster and more stable learning of the `\\boxed{}` answer format.
4. **Memory:** Both runs peaked near 80GB, but neither OOM'd.

### Hypothesis Assessment

The noise-control hypothesis is **supported** by these results:
- Track B (trained with trivia noise) generalized better to clean test data than Track A.
- The noise may have acted as a regularizer, forcing the model to attend more carefully to task-relevant signal.

### Recommendations

1. **Investigate late training dynamics:** Confirmed that the validation gap persists all the way to 400 steps.
2. **Consider ablations:** Test different noise types (e.g., random words vs. factual sentences).
3. **Publish findings:** The consistent effect size and robustness suggest this is a meaningful result.

In [ ]:
# Create summary comparison table
summary_data = []

# Overall statistics
for track, df, name in [('A', df_a_aligned, 'Clean'), ('B', df_b_aligned, 'Mixed')]:
    row = {
        'Track': name,
        'Total Steps': df['_step'].max(),
        'Final Val Accuracy': df['val/HuggingFaceH4/MATH-500/reward/mean'].dropna().iloc[-1] if not df['val/HuggingFaceH4/MATH-500/reward/mean'].dropna().empty else np.nan,
        'Best Val Accuracy': df['val/HuggingFaceH4/MATH-500/reward/mean'].max(),
        'Mean Response Length': df['response_length/mean'].mean(),
        'Max Response Length': df['response_length/mean'].max(),
        'Mean Clip Ratio': df['response_length/clip_ratio'].mean(),
        'Mean Time/Step (s)': df['perf/time_per_step'].mean(),
        'Mean Reward': df['critic/rewards/mean'].mean(),
        'Mean Entropy': df['actor/entropy'].mean(),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('1p7_hendrycks_summary_table.csv', index=False)

print("Track Comparison Summary:")
print("="*100)
for col in summary_df.columns:
    if col != 'Track':
        print(f"\n{col}:")
        for _, row in summary_df.iterrows():
            val = row[col]
            if isinstance(val, float):
                print(f"  {row['Track']:<10}: {val:.4f}")
            else:
                print(f"  {row['Track']:<10}: {val}")

# Display the table
summary_df

## 8. Per-Question Answer Evolution (Training-Run Rollouts Only)

This section drills into the **Response Length Dynamics** finding using only the validation rollouts saved during training at common checkpoints: **steps 0, 100, 200, 300, and 400**. No answers are regenerated; all text comes from the saved `{step}_rollouts.json` files in the Modal volume `e3-generation-vol`.

### Key Finding to Investigate
With Track A running to full completion (step 400), we can see the full format adoption and accuracy progression. Track B learned the `\\boxed{}` answer format much faster, maintaining lower clip ratios throughout the run. The cells below explore *how* this unfolded at the per-question level, comparing responses at **step 0** versus **step 400**.

In [ ]:
import subprocess

ROLLOUT_DIR = "rollouts_1p7b_hendrycks"
TRACK_CONFIGS = {
    "track_a_hendrycks": "qwen3-1p7b-hendrycks-grpo-clean",
    "track_b_hendrycks": "qwen3-1p7b-hendrycks-grpo-mixed",
}
STEPS = [0, 100, 200, 300, 400]
EXP_NAME_A = TRACK_CONFIGS["track_a_hendrycks"]
EXP_NAME_B = TRACK_CONFIGS["track_b_hendrycks"]

missing = []
for track_key, exp_name in TRACK_CONFIGS.items():
    track_dir = os.path.join(ROLLOUT_DIR, track_key)
    os.makedirs(track_dir, exist_ok=True)
    for step in STEPS:
        local_path = os.path.join(track_dir, f"{step}_rollouts.json")
        if not os.path.exists(local_path):
            missing.append((track_key, exp_name, step, local_path))

if missing:
    print(f"Missing {len(missing)} rollout files. Downloading from Modal volume 'e3-generation-vol'...")
    for track_key, exp_name, step, local_path in missing:
        track_dir = os.path.dirname(local_path)
        remote_path = f"ckpts/{exp_name}/{step}_rollouts.json"
        cmd = ["modal", "volume", "get", "e3-generation-vol", remote_path]
        print(f"  Downloading {remote_path} -> {local_path}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=track_dir)
        if result.returncode != 0:
            print(f"ERROR: {result.stderr}")
            raise RuntimeError(f"Failed to download {remote_path}")
    print("Download complete.")
else:
    print("All rollout files already present locally.")

# Verify
for track_key, exp_name in TRACK_CONFIGS.items():
    for step in STEPS:
        local_path = os.path.join(ROLLOUT_DIR, track_key, f"{step}_rollouts.json")
        assert os.path.exists(local_path), f"Missing: {local_path}"
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"  OK: {track_key}/{step}_rollouts.json ({size_mb:.1f} MiB)")
print("All rollouts ready.")

## 8.1 Build the Tidy Rollout DataFrame

We load every saved rollout JSON into a tidy DataFrame (`df_rollouts`) using the correct keys (`index`, `input`, `output`, `score`) and aggregate to question level (`df_questions`, pass@1). All qualitative and quantitative rollout analyses below use these two tables.

In [ ]:
import json
import pandas as pd
from collections import defaultdict

ROLLOUT_DIR = "rollouts_1p7b_hendrycks"

def load_rollouts(track, step):
    path = os.path.join(ROLLOUT_DIR, track, f"{step}_rollouts.json")
    with open(path) as f:
        return json.load(f)

# Load all rollouts into a tidy DataFrame
rows = []
for track, label in [("track_a_hendrycks", "A"), ("track_b_hendrycks", "B")]:
    for step in STEPS:
        data = load_rollouts(track, step)
        for r in data:
            rows.append({
                "track": label,
                "step": step,
                "question_idx": r["index"],
                "input_text": r["input"],
                "output_text": r["output"],
                "score": r["score"],
                "is_correct": r["score"] == 1.0,
                "response_length": len(r["output"]),
                "has_hash": "\\boxed{" in r["output"],
            })

df_rollouts = pd.DataFrame(rows)
print(f"Loaded {len(df_rollouts)} rollout samples")

# Aggregate to question level (pass@1: at least one sample correct)
df_questions = df_rollouts.groupby(["track", "step", "question_idx"]).agg(
    is_correct=("is_correct", "max"),
    response_length_mean=("response_length", "mean"),
    has_any_hash=("has_hash", "max"),
    n_samples=("score", "count"),
).reset_index()

print("\nQuestion-level accuracy by track and step:")
print(df_questions.groupby(["track", "step"])["is_correct"].mean().unstack())

### Both Wrong

Questions neither track could solve by the final step 400 — genuinely hard problems. We display step 0 vs. step 400 outputs to observe long-term evolution.

In [ ]:
import random
from IPython.display import display, Markdown
from html import escape

# Find questions both tracks got wrong at step 400
step_final = df_questions[df_questions["step"] == 400]
wrong_a = set(step_final[(step_final["track"] == "A") & (~step_final["is_correct"])]["question_idx"])
wrong_b = set(step_final[(step_final["track"] == "B") & (~step_final["is_correct"])]["question_idx"])
both_wrong = sorted(wrong_a & wrong_b)

display(Markdown(f"**Questions both tracks got WRONG at step 400:** {len(both_wrong)} / {len(step_final[step_final['track']=='A'])}"))

if both_wrong:
    random.seed()
    q_idx = random.choice(both_wrong)
    
    # Extract the question text from the first available sample
    sample_input = df_rollouts[df_rollouts["question_idx"] == q_idx]["input_text"].iloc[0]
    question_text = sample_input.replace(" Let's think step by step and output the final answer within \"\\boxed{}\".", "").replace("user\n", "").replace("\nassistant\n", "").strip()
    
    display(Markdown("---"))
    display(Markdown(f"## Question (idx={q_idx})"))
    display(Markdown(question_text))
    
    for track in ["A", "B"]:
        display(Markdown("---"))
        display(Markdown(f"## Track {track}"))
        for step in [0, 400]:
            subset = df_rollouts[(df_rollouts["track"] == track) & (df_rollouts["step"] == step) & (df_rollouts["question_idx"] == q_idx)]
            if len(subset) == 0:
                continue
            # Show the first sample for brevity
            row = subset.iloc[0]
            
            display(Markdown(
                f"**Step {step}** &nbsp;&middot;&nbsp; "
                f"length={row['response_length']} chars &nbsp;&middot;&nbsp; "
                f"correct={row['is_correct']} &nbsp;&middot;&nbsp; "
                f"has_boxed={row['has_hash']}"
            ))
            
            text = str(row["output_text"])
            html = (
                f'<div style="background:#f8f8f8; padding:10px; border-radius:4px; '
                f'border:1px solid #ddd; font-family:monospace; font-size:13px;">'
                f'<pre style="white-space:pre-wrap; word-wrap:break-word; margin:0; color:black">'
                f'{escape(text)}</pre></div>'
            )
            display(Markdown(html))
else:
    display(Markdown("No questions found where both tracks are wrong at step 400."))

### Both Right

Questions both tracks solved by step 400. Even when both are correct, Track B may produce shorter reasoning; we surface the example where Track B was most concise relative to Track A.

In [ ]:
# Find questions both tracks got right at step 400
step_final = df_questions[df_questions["step"] == 400]
right_a = set(step_final[(step_final["track"] == "A") & (step_final["is_correct"])]["question_idx"])
right_b = set(step_final[(step_final["track"] == "B") & (step_final["is_correct"])]["question_idx"])
both_right = sorted(right_a & right_b)

display(Markdown(f"**Questions both tracks got RIGHT at step 400:** {len(both_right)} / {len(step_final[step_final['track']=='A'])}"))

if both_right:
    # Let's find one where Track B was substantially shorter at step 400
    candidates = []
    for q_idx in both_right:
        len_a = df_rollouts[(df_rollouts["track"] == "A") & (df_rollouts["step"] == 400) & (df_rollouts["question_idx"] == q_idx)]["response_length"].mean()
        len_b = df_rollouts[(df_rollouts["track"] == "B") & (df_rollouts["step"] == 400) & (df_rollouts["question_idx"] == q_idx)]["response_length"].mean()
        candidates.append((q_idx, len_a - len_b))
    
    # Sort by how much shorter B was compared to A
    candidates.sort(key=lambda x: x[1], reverse=True)
    q_idx = candidates[0][0]
    
    # Extract the question text
    sample_input = df_rollouts[df_rollouts["question_idx"] == q_idx]["input_text"].iloc[0]
    question_text = sample_input.replace(" Let's think step by step and output the final answer within \"\\boxed{}\".", "").replace("user\n", "").replace("\nassistant\n", "").strip()
    
    display(Markdown("---"))
    display(Markdown(f"## Question (idx={q_idx})"))
    display(Markdown(question_text))
    
    for track in ["A", "B"]:
        display(Markdown("---"))
        display(Markdown(f"## Track {track}"))
        for step in [0, 400]:
            subset = df_rollouts[(df_rollouts["track"] == track) & (df_rollouts["step"] == step) & (df_rollouts["question_idx"] == q_idx)]
            if len(subset) == 0:
                continue
            row = subset.iloc[0]
            
            display(Markdown(
                f"**Step {step}** &nbsp;&middot;&nbsp; "
                f"length={row['response_length']} chars &nbsp;&middot;&nbsp; "
                f"correct={row['is_correct']} &nbsp;&middot;&nbsp; "
                f"has_boxed={row['has_hash']}"
            ))
            
            text = str(row["output_text"])
            html = (
                f'<div style="background:#f8f8f8; padding:10px; border-radius:4px; '
                f'border:1px solid #ddd; font-family:monospace; font-size:13px;">'
                f'<pre style="white-space:pre-wrap; word-wrap:break-word; margin:0; color:black">'
                f'{escape(text)}</pre></div>'
            )
            display(Markdown(html))
else:
    display(Markdown("No questions found where both tracks are right at step 400."))

### Track A Right, Track B Wrong

Rare cases at step 400 where Track A's clean-training pathway succeeded while the mixed-noise Track B failed.

In [ ]:
# Find questions Track A got right and Track B got wrong at step 400
step_final = df_questions[df_questions["step"] == 400]
right_a = set(step_final[(step_final["track"] == "A") & (step_final["is_correct"])]["question_idx"])
wrong_b = set(step_final[(step_final["track"] == "B") & (~step_final["is_correct"])]["question_idx"])
a_right_b_wrong = sorted(right_a & wrong_b)

display(Markdown(f"**Questions Track A got RIGHT but Track B got WRONG at step 400:** {len(a_right_b_wrong)} / {len(step_final[step_final['track']=='A'])}"))

if a_right_b_wrong:
    random.seed(42)
    q_idx = random.choice(a_right_b_wrong)
    
    # Extract the question text
    sample_input = df_rollouts[df_rollouts["question_idx"] == q_idx]["input_text"].iloc[0]
    question_text = sample_input.replace(" Let's think step by step and output the final answer within \"\\boxed{}\".", "").replace("user\n", "").replace("\nassistant\n", "").strip()
    
    display(Markdown("---"))
    display(Markdown(f"## Question (idx={q_idx})"))
    display(Markdown(question_text))
    
    for track in ["A", "B"]:
        display(Markdown("---"))
        display(Markdown(f"## Track {track}"))
        for step in [0, 400]:
            subset = df_rollouts[(df_rollouts["track"] == track) & (df_rollouts["step"] == step) & (df_rollouts["question_idx"] == q_idx)]
            if len(subset) == 0:
                continue
            row = subset.iloc[0]
            
            display(Markdown(
                f"**Step {step}** &nbsp;&middot;&nbsp; "
                f"length={row['response_length']} chars &nbsp;&middot;&nbsp; "
                f"correct={row['is_correct']} &nbsp;&middot;&nbsp; "
                f"has_boxed={row['has_hash']}"
            ))
            
            text = str(row["output_text"])
            html = (
                f'<div style="background:#f8f8f8; padding:10px; border-radius:4px; '
                f'border:1px solid #ddd; font-family:monospace; font-size:13px;">'
                f'<pre style="white-space:pre-wrap; word-wrap:break-word; margin:0; color:black">'
                f'{escape(text)}</pre></div>'
            )
            display(Markdown(html))
else:
    display(Markdown("No questions found where Track A is right and Track B is wrong at step 400."))

### Track B Right, Track A Wrong

Cases at step 400 where Track B succeeded while Track A failed, showing the mixed-noise environment was not a hindrance.

In [ ]:
# Find questions Track B got right and Track A got wrong at step 400
step_final = df_questions[df_questions["step"] == 400]
wrong_a = set(step_final[(step_final["track"] == "A") & (~step_final["is_correct"])]["question_idx"])
right_b = set(step_final[(step_final["track"] == "B") & (step_final["is_correct"])]["question_idx"])
b_right_a_wrong = sorted(wrong_a & right_b)

display(Markdown(f"**Questions Track B got RIGHT but Track A got WRONG at step 400:** {len(b_right_a_wrong)} / {len(step_final[step_final['track']=='A'])}"))

if b_right_a_wrong:
    random.seed()
    q_idx = random.choice(b_right_a_wrong)
    
    # Extract the question text
    sample_input = df_rollouts[df_rollouts["question_idx"] == q_idx]["input_text"].iloc[0]
    question_text = sample_input.replace(" Let's think step by step and output the final answer within \"\\boxed{}\".", "").replace("user\n", "").replace("\nassistant\n", "").strip()
    
    display(Markdown("---"))
    display(Markdown(f"## Question (idx={q_idx})"))
    display(Markdown(question_text))
    
    for track in ["A", "B"]:
        display(Markdown("---"))
        display(Markdown(f"## Track {track}"))
        for step in [0, 400]:
            subset = df_rollouts[(df_rollouts["track"] == track) & (df_rollouts["step"] == step) & (df_rollouts["question_idx"] == q_idx)]
            if len(subset) == 0:
                continue
            row = subset.iloc[0]
            
            display(Markdown(
                f"**Step {step}** &nbsp;&middot;&nbsp; "
                f"length={row['response_length']} chars &nbsp;&middot;&nbsp; "
                f"correct={row['is_correct']} &nbsp;&middot;&nbsp; "
                f"has_boxed={row['has_hash']}"
            ))
            
            text = str(row["output_text"])
            html = (
                f'<div style="background:#f8f8f8; padding:10px; border-radius:4px; '
                f'border:1px solid #ddd; font-family:monospace; font-size:13px;">'
                f'<pre style="white-space:pre-wrap; word-wrap:break-word; margin:0; color:black">'
                f'{escape(text)}</pre></div>'
            )
            display(Markdown(html))
else:
    display(Markdown("No questions found where Track B is right and Track A is wrong at step 400."))

### Format Adoption and Accuracy

These plots track two things over time: (left) what percentage of questions have `####` in at least one sample, and (right) the validation accuracy. Format adoption rising before or alongside accuracy suggests the model first learned the output format, then leveraged it to improve reasoning.

In [ ]:
import matplotlib.pyplot as plt

# Compute adoption stats per (track, step)
adoption = df_questions.groupby(["track", "step"]).agg(
    hash_rate=("has_any_hash", "mean"),
    accuracy=("is_correct", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: % of questions with #### in any sample
ax = axes[0]
for track, color in [("A", "#e74c3c"), ("B", "#2ecc71")]:
    d = adoption[adoption["track"] == track]
    ax.plot(d["step"], d["hash_rate"] * 100, "o-", label=f"Track {track}", linewidth=2, markersize=8, color=color)
ax.set_xlabel("Training Step")
ax.set_ylabel("% Questions with \\boxed{} in any sample")
ax.set_title("\\boxed{} Format Adoption")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy for comparison
ax = axes[1]
for track, color in [("A", "#e74c3c"), ("B", "#2ecc71")]:
    d = adoption[adoption["track"] == track]
    ax.plot(d["step"], d["accuracy"] * 100, "s-", label=f"Track {track}", linewidth=2, markersize=8, color=color)
ax.set_xlabel("Training Step")
ax.set_ylabel("% Questions Correct (pass@1)")
ax.set_title("Validation Accuracy (from Rollouts)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle("Format Adoption vs Accuracy (Steps 0, 100, 200, 300, 400)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("format_adoption.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nAdoption table:")
print(adoption.to_string(index=False))

### Response Length Distribution by Correctness

These histograms compare the length of correct versus incorrect answers at each step. If wrong answers tend to be much longer than correct ones, the model may be rambling when confused. Track B's distributions should show less separation between correct and wrong lengths, suggesting more consistent formatting.

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 20))
 
steps = STEPS
colors = {"A": "#e74c3c", "B": "#2ecc71"}
bins = range(0, 4000, 200)
 
for row, step in enumerate(steps):
    data = df_questions[df_questions["step"] == step]
    
    for col, (title, mask) in enumerate([
        ("All", slice(None)),
        ("Correct", data["is_correct"]),
        ("Wrong", ~data["is_correct"]),
    ]):
        ax = axes[row][col]
        subset = data[mask] if mask is slice(None) else data[mask]
        for track in ["A", "B"]:
            t = subset[subset["track"] == track]
            ax.hist(t["response_length_mean"], bins=bins, alpha=0.5, color=colors[track], label=f"Track {track}", density=True)
        ax.set_title(f"Step {step} — {title}")
        ax.set_xlabel("Response Length (chars)")
        ax.set_ylabel("Density")
        ax.legend()
        ax.grid(True, alpha=0.3)
 
plt.suptitle("Response Length Distribution by Correctness", fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig("length_distribution_by_correctness.png", dpi=150, bbox_inches="tight")
plt.show()
 
# Print means
print("\nMean response length by correctness:")
for track in ["A", "B"]:
    for step in steps:
        subset = df_questions[(df_questions["track"] == track) & (df_questions["step"] == step)]
        c_mean = subset[subset["is_correct"]]["response_length_mean"].mean()
        w_mean = subset[~subset["is_correct"]]["response_length_mean"].mean()
        print(f"Track {track} step {step}: correct={c_mean:.1f}, wrong={w_mean:.1f}, delta={w_mean-c_mean:.1f}")

### Efficiency Trajectory: Accuracy vs. Response Length

This plot shows the trade-off between how accurate each track is and how long its responses are. The ideal direction is toward the top-left: high accuracy with short responses. Arrows show the progression from step 0 to 100 to 200, revealing how each track's efficiency evolved during training.

In [ ]:
# Compute per-step aggregates
eff = df_questions.groupby(["track", "step"]).agg(
    accuracy=("is_correct", "mean"),
    mean_length=("response_length_mean", "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for track, color in [("A", "#e74c3c"), ("B", "#2ecc71")]:
    d = eff[eff["track"] == track].sort_values("step")
    ax.plot(d["mean_length"], d["accuracy"] * 100, "o-", label=f"Track {track}", linewidth=2, markersize=8, color=color)
    # Add step annotations
    for _, row in d.iterrows():
        ax.annotate(f"  s{int(row['step'])}", 
                    xy=(row["mean_length"], row["accuracy"] * 100),
                    fontsize=9, color=color)
    # Draw arrows between steps
    for i in range(len(d) - 1):
        ax.annotate("", xy=(d.iloc[i+1]["mean_length"], d.iloc[i+1]["accuracy"] * 100),
                    xytext=(d.iloc[i]["mean_length"], d.iloc[i]["accuracy"] * 100),
                    arrowprops=dict(arrowstyle="->", color=color, lw=1.5, alpha=0.6))

ax.set_xlabel("Mean Response Length (chars)", fontsize=12)
ax.set_ylabel("Validation Accuracy (%)", fontsize=12)
ax.set_title("Efficiency Trajectory: Accuracy vs. Response Length", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Ideal direction annotation
ax.annotate("Better ->", xy=(0.85, 0.85), xycoords="axes fraction",
            fontsize=14, color="green", fontweight="bold", ha="center")

plt.tight_layout()
plt.savefig("efficiency_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nEfficiency data:")
print(eff.to_string(index=False))

### Answer Position Over Time

This plot tracks where the `####` delimiter appears within correct answers. An earlier position (higher on the inverted y-axis) means the model is giving the final answer sooner, indicating more concise reasoning. Track B's line trending upward suggests it learned to place the answer earlier in the response.

In [ ]:
import re

def find_hash_position(text):
    m = re.search(r"####", text)
    return m.start() if m else None

# Compute hash position for each rollout sample
df_rollouts["hash_position"] = df_rollouts["output_text"].apply(find_hash_position)

# Average position among correct answers that contain ####
pos_stats = df_rollouts[df_rollouts["is_correct"] & df_rollouts["has_hash"]].groupby(["track", "step"])["hash_position"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))

for track, color in [("A", "#e74c3c"), ("B", "#2ecc71")]:
    d = pos_stats[pos_stats["track"] == track].sort_values("step")
    ax.plot(d["step"], d["hash_position"], "o-", label=f"Track {track}", linewidth=2, markersize=8, color=color)

ax.set_xlabel("Training Step")
ax.set_ylabel("Avg Character Position of ####")
ax.set_title("Answer Position Over Time (Correct Answers Only)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.invert_yaxis()  # Earlier position = higher on chart = better

plt.tight_layout()
plt.savefig("answer_position.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nAnswer position stats:")
print(pos_stats.to_string(index=False))

In [ ]:
# Final experiment summary printout

print("="*80)
print("QWEN3-1.7B HENDRYCKS MATH 400-STEP RLAD NOISE-CONTROL EXPERIMENT SUMMARY")
print("="*80)

print("\n## Training Configuration")
print(f"Model: Qwen/Qwen3-1.7B")
print(f"Track A (Clean): {len(df_a)} steps")
print(f"Track B (Mixed): {len(df_b)} steps")

print("\n## Key Findings")

# Final validation accuracy
if 'val/HuggingFaceH4/MATH-500/reward/mean' in df_a.columns:
    final_reward_a = df_a['val/HuggingFaceH4/MATH-500/reward/mean'].iloc[-1]
    final_reward_b = df_b['val/HuggingFaceH4/MATH-500/reward/mean'].iloc[-1]
    print(f"\nFinal Validation Accuracy:")
    print(f"  Track A (Clean): {final_reward_a:.4f}")
    print(f"  Track B (Mixed): {final_reward_b:.4f}")
    print(f"  Difference: {final_reward_a - final_reward_b:.4f}")

# Response length evolution
if 'response_length/mean' in df_a.columns:
    initial_len_a = df_a['response_length/mean'].iloc[0]
    final_len_a = df_a['response_length/mean'].iloc[-1]
    initial_len_b = df_b['response_length/mean'].iloc[0]
    final_len_b = df_b['response_length/mean'].iloc[-1]
    
    print(f"\nResponse Length Evolution:")
    print(f"  Track A: {initial_len_a:.1f} → {final_len_a:.1f} (Δ={final_len_a-initial_len_a:.1f})")
    print(f"  Track B: {initial_len_b:.1f} → {final_len_b:.1f} (Δ={final_len_b-initial_len_b:.1f})")

# Training stability
print(f"\nTraining Stability:")
print(f"  Both tracks completed all 400 steps without interruption")
print(f"  Track A run ID: 5ot5gmgq (finished)")
print(f"  Track B run ID: 00h4z032 (finished)")

print("\n" + "="*80)
print("Analysis complete! All artifacts saved with '1p7_hendrycks_' prefix.")
print("="*80)

In [ ]:
# Save detailed comparison table (1p7_hendrycks_summary_table.csv already written in Section 7)
detailed_summary = []
for step in STEPS:
    if step <= max_step:
        row_a = df_a[df_a['_step'] == step]
        row_b = df_b[df_b['_step'] == step]

        if not row_a.empty and not row_b.empty:
            detailed_summary.append({
                'Step': step,
                'Track A Val Reward': row_a['val/HuggingFaceH4/MATH-500/reward/mean'].values[0] if 'val/HuggingFaceH4/MATH-500/reward/mean' in row_a.columns else None,
                'Track B Val Reward': row_b['val/HuggingFaceH4/MATH-500/reward/mean'].values[0] if 'val/HuggingFaceH4/MATH-500/reward/mean' in row_b.columns else None,
                'Track A Resp Length': row_a['response_length/mean'].values[0] if 'response_length/mean' in row_a.columns else None,
                'Track B Resp Length': row_b['response_length/mean'].values[0] if 'response_length/mean' in row_b.columns else None,
            })

detailed_df = pd.DataFrame(detailed_summary)
detailed_df.to_csv('1p7_hendrycks_detailed_comparison.csv', index=False)
print("Saved detailed comparison to 1p7_hendrycks_detailed_comparison.csv")

print("\nDetailed Comparison Table:")
print(detailed_df.to_string(index=False))

In [ ]:
# Print final data availability summary
print("="*80)
print("DATA AVAILABILITY SUMMARY")
print("="*80)

print("\nWandB History Data:")
print(f"  Track A (5ot5gmgq): {len(df_a)} steps")
print(f"  Track B (00h4z032): {len(df_b)} steps")

print("\nModal Rollout Data:")
for label, exp in [("A", "qwen3-1p7b-hendrycks-grpo-clean"), ("B", "qwen3-1p7b-hendrycks-grpo-mixed")]:
    available_steps = sorted(df_rollouts[df_rollouts["track"] == label]["step"].unique().tolist())
    missing_steps = [s for s in STEPS if s not in available_steps]
    print(f"  {exp}:")
    print(f"    Available steps: {available_steps}")
    if missing_steps:
        print(f"    Missing steps: {missing_steps}")
    else:
        print(f"    All steps available!")

print("\nGenerated Outputs:")
print("  - 1p7_hendrycks_analysis_complete_run.ipynb (this notebook)")
print("  - 1p7_hendrycks_summary_table.csv")
print("  - 1p7_hendrycks_detailed_comparison.csv")
print("  - 1p7_hendrycks_response_length_comparison.png")
print("  - 1p7_hendrycks_reward_comparison.png")
print("  - 1p7_hendrycks_per_step_time.png")

print("\n" + "="*80)
print("Analysis notebook is now self-sufficient with embedded WandB/Modal fetching.")
print("="*80)

## 9. Pass@k Sampling-Budget Comparison (up to pass@32)

The Section 8 rollouts were saved during training with only **4 samples per question** (`val_kwargs.n=4`), so they can show at most pass@4. To study how each track scales with a larger sampling budget, we ran a dedicated evaluation that generates **32 samples per question** on the full MATH-500 test set for the **step-400** checkpoint of each track (see `scripts/run_pass32_sweep_1p7_hendrycks.sh`, which scores the already-converted HF checkpoint with `modal_eval_general.py`).

Each eval writes a `per_problem_*.csv` (with `n_correct` out of `n_samples=32` per question). From those counts we compute the **unbiased pass@k estimator** (Chen et al., HumanEval) for `k in {1, 2, 4, 8, 16, 32}` and compare Track A (clean) vs Track B (mixed).

We evaluate only the **step-400** checkpoint of each track (no base-model eval).

In [ ]:
# Download the 32-sample per-problem eval CSVs from the Modal volume.
# These are produced by scripts/run_pass32_sweep_1p7_hendrycks.sh and live under
# /data/pass32_1p7b_hendrycks/ on volume 'e3-generation-vol'.
import os
import subprocess

PASS_DIR = "pass32_1p7b_hendrycks"          # local cache
PASS_STEPS = [400]   # only the step-400 checkpoints were evaluated
KS = [1, 2, 4, 8, 16, 32]
os.makedirs(PASS_DIR, exist_ok=True)


def remote_csv_name(track, step):
    return f"per_problem_math_{track}_step{step}.csv"


missing = []
for track in ("track_a_hendrycks", "track_b_hendrycks"):
    for step in PASS_STEPS:
        fname = remote_csv_name(track, step)
        local_path = os.path.join(PASS_DIR, fname)
        if not os.path.exists(local_path):
            missing.append((fname, local_path))

# De-duplicate (the base step-0 file is shared by both tracks).
missing = list({f: (f, p) for f, p in missing}.values())

if missing:
    print(f"Downloading {len(missing)} eval CSVs from Modal volume 'e3-generation-vol'...")
    for fname, local_path in missing:
        remote_path = f"pass32_1p7b_hendrycks/{fname}"
        cmd = ["modal", "volume", "get", "e3-generation-vol", remote_path]
        print(f"  {remote_path} -> {local_path}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=PASS_DIR)
        if result.returncode != 0:
            print(f"  ERROR: {result.stderr.strip()}")
            raise RuntimeError(f"Failed to download {remote_path}. Has the sweep finished?")
    print("Download complete.")
else:
    print("All pass@k eval CSVs already present locally.")

# Verify presence and sample size
for track in ("track_a_hendrycks", "track_b_hendrycks"):
    for step in PASS_STEPS:
        p = os.path.join(PASS_DIR, remote_csv_name(track, step))
        assert os.path.exists(p), f"Missing: {p}"
print("All pass@k eval CSVs ready.")

In [ ]:
# Compute the unbiased pass@k for k in {1,2,4,8,16,32} from per-problem n_correct.
import numpy as np
import pandas as pd


def pass_at_k(n, c, k):
    """Unbiased pass@k estimator (Chen et al., HumanEval).

    n = samples per question, c = number correct, k = budget.
    Probability that at least one of k drawn samples is correct.
    """
    if n - c < k:
        return 1.0
    return float(1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))


rows = []
for track, label in (("track_a_hendrycks", "A"), ("track_b_hendrycks", "B")):
    for step in PASS_STEPS:
        df = pd.read_csv(os.path.join(PASS_DIR, remote_csv_name(track, step)))
        n_samples = int(df["n_samples"].iloc[0])
        assert (df["n_samples"] == n_samples).all(), "inconsistent n_samples in CSV"
        num_problems = len(df)
        for k in KS:
            if k > n_samples:
                continue
            vals = [pass_at_k(n_samples, int(c), k) for c in df["n_correct"]]
            rows.append({
                "track": label,
                "step": step,
                "k": k,
                "pass_at_k": float(np.mean(vals)),
                "n_samples": n_samples,
                "num_problems": num_problems,
            })

df_passk = pd.DataFrame(rows)

# Sanity check: pass@k must be non-decreasing in k for each (track, step).
for (t, s), g in df_passk.groupby(["track", "step"]):
    g = g.sort_values("k")
    assert (g["pass_at_k"].diff().dropna() >= -1e-9).all(), f"pass@k not monotonic for track {t} step {s}"

print(f"Computed pass@k for {df_passk[['track','step']].drop_duplicates().shape[0]} (track, step) pairs.")
print(f"Samples per question: {sorted(df_passk['n_samples'].unique())}, "
      f"problems: {sorted(df_passk['num_problems'].unique())}")
df_passk.head(12)

In [ ]:
# Pass@k sweep curves: one subplot per checkpoint step, Track A vs Track B.
import matplotlib.pyplot as plt

steps = sorted(df_passk["step"].unique())
colors = {"A": "#e74c3c", "B": "#2ecc71"}

ncols = 4
nrows = int(np.ceil(len(steps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), squeeze=False)

for idx, step in enumerate(steps):
    ax = axes[idx // ncols][idx % ncols]
    for label in ("A", "B"):
        d = df_passk[(df_passk["track"] == label) & (df_passk["step"] == step)].sort_values("k")
        ax.plot(d["k"], d["pass_at_k"] * 100, "o-", color=colors[label],
                linewidth=2, markersize=6, label=f"Track {label}")
    ax.set_xscale("log", base=2)
    ax.set_xticks(KS)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel("k (samples)")
    ax.set_ylabel("pass@k (%)")
    ax.set_title(f"Step {step}")
    ax.grid(True, alpha=0.3)
    ax.legend()

# Hide any unused subplots
for j in range(len(steps), nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")

plt.suptitle("Pass@k Sampling-Budget Comparison (k = 1, 2, 4, 8, 16, 32)",
             fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("pass_at_k_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

# Overlay: pass@32 (and pass@1) over training steps for both tracks.
fig, ax = plt.subplots(figsize=(9, 5))
for label in ("A", "B"):
    for k, style in ((1, "--"), (32, "o-")):
        d = df_passk[(df_passk["track"] == label) & (df_passk["k"] == k)].sort_values("step")
        ax.plot(d["step"], d["pass_at_k"] * 100, style, color=colors[label],
                linewidth=2, markersize=6, label=f"Track {label} pass@{k}")
ax.set_xlabel("Training Step")
ax.set_ylabel("Accuracy (%)")
ax.set_title("pass@1 vs pass@32 over Training")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("pass_at_1_vs_32_over_steps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table: pass@k columns x (track, step) rows, plus A-vs-B gap at pass@32.
pivot = (
    df_passk
    .pivot_table(index=["step", "track"], columns="k", values="pass_at_k")
    .sort_index()
)
pivot.columns = [f"pass@{k}" for k in pivot.columns]
pivot = (pivot * 100).round(2)
pivot.to_csv("1p7_hendrycks_pass_at_k.csv")
print("Saved pass@k table to 1p7_hendrycks_pass_at_k.csv\n")
print(pivot.to_string())

# Track B minus Track A at pass@32, per step.
p32 = df_passk[df_passk["k"] == 32].pivot_table(index="step", columns="track", values="pass_at_k")
if {"A", "B"}.issubset(p32.columns):
    gap = ((p32["B"] - p32["A"]) * 100).round(2)
    print("\npass@32 gap (Track B - Track A), percentage points:")
    print(gap.to_string())